[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C18/E18-2_Risque_Panne.ipynb)

# Risque de panne d'un réducteur : une étude de classification de bout en bout

## Le contexte

Un exploitant suit un parc de 1 600 réducteurs et boîtes de vitesses. Pour chacun, l'équipe de maintenance a relevé une fois l'âge, la charge, la vitesse, le couple, le temps écoulé depuis le dernier graissage, la contamination de l'huile, sa température et la vibration du carter, puis a classé l'équipement dans une classe de risque de panne : faible, moyen ou élevé. L'exploitant voudrait classer les équipements sans passer par une inspection, pour programmer les interventions sur ceux qui en ont besoin.

La grandeur à prédire est une classe parmi trois : c'est un problème de classification, déséquilibré. La classe élevée, la plus utile à repérer, est la plus rare : 9 % des équipements.

Défi : prédire `classe_risque` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Le F-score macro est la moyenne des F-scores des trois classes : il pèse autant la classe rare que la classe fréquente. Le modèle de départ est une régression logistique sur les relevés mis à l'échelle ; il atteint 0,695. Deux comparaisons sans apprentissage : toujours répondre « faible » (0,255) et une règle de vibration par seuils tirés de l'apprentissage (0,448). À vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Risque_Panne.xlsx`, dans le dossier `donnees` du dépôt : 1 600 réducteurs, 11 colonnes. Une ligne est un réducteur du parc, relevé une fois.

| Colonne | Signification | Unité |
|---|---|---|
| `type_actif` | type d'équipement : `standard`, `lourd` ou `boite` (boîte de vitesses) | catégorie |
| `age_annees` | durée de service depuis la mise en route | années |
| `taux_charge` | charge moyenne rapportée à la charge nominale | sans unité |
| `vitesse_rpm` | vitesse de rotation moyenne de l'arbre | tr/min |
| `couple_Nm` | couple moyen transmis | N·m |
| `puissance_kW` | puissance calculée à partir du couple et de la vitesse (colonne redondante) | kW |
| `heures_depuis_graissage` | heures de marche depuis le dernier graissage | h |
| `contamination_ppm` | particules dans l'huile (quelques cases vides) | ppm |
| `temperature_huile_C` | température de l'huile (quelques cases vides) | °C |
| `vibration_mm_s` | vitesse de vibration efficace au carter | mm/s |
| `classe_risque` | cible : classe de risque de panne, `faible`, `moyen` ou `eleve`, établie par l'équipe de maintenance | catégorie |

Origine : jeu simulé, une usure cachée croît avec l'âge, la charge, le temps depuis le graissage, la contamination de l'huile et une bande de résonance en vitesse, et fixe la classe de risque ; la température et la vibration n'en portent qu'une faible trace. Il enseigne une méthode, et aucune conclusion de maintenance n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, ses types et la répartition des classes : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Risque_Panne.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Risque_Panne.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "classe_risque"
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
print("types :", df.dtypes.astype(str).value_counts().to_dict())
repartition = df[cible].value_counts()
print("répartition des classes :", repartition.to_dict())
df.head(3)

Lecture du résultat : 1 600 réducteurs, 11 colonnes. Deux colonnes ont des cases vides : la contamination (68) et la température de l'huile (50). Les classes sont déséquilibrées : 992 équipements à risque faible, 464 à risque moyen, 144 à risque élevé. Un modèle qui répondrait toujours « faible » aurait une exactitude de 62 %, sans repérer un seul équipement à risque : l'exactitude ne dit rien ici, le F-score macro le dit.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Risque_Panne.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des réducteurs est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. Le tirage est stratifié sur la classe : chaque classe garde sa part dans l'apprentissage et dans le test, ce qui garantit des cas de la classe rare dans la base de test. La graine 42 fixe le tirage ; chacun obtient la même base de test, et les scores se comparent.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "réducteurs ; test :", len(X_test), "réducteurs")
print("classe élevée dans le test :", int((y_test == "eleve").sum()), "réducteurs")

### Étape 3 : explorer l'apprentissage

Trois lectures sur `X_train` et `y_train` : les médianes de chaque relevé par classe de risque, la dispersion de la vitesse par classe (écart entre le premier et le troisième quartile), et la corrélation entre le couple, la vitesse et la puissance calculée, pour repérer une colonne redondante.

In [ ]:
numeriques = X_train.select_dtypes("number").columns
par_classe = X_train[numeriques].groupby(y_train).median().loc[["faible", "moyen", "eleve"]]
print(par_classe.round(2).T.to_string())
quartiles = X_train["vitesse_rpm"].groupby(y_train).quantile([0.25, 0.75]).unstack().loc[["faible", "moyen", "eleve"]]
print(quartiles.round(0).to_string())
puissance_calculee = X_train["couple_Nm"] * X_train["vitesse_rpm"] * 2 * np.pi / 60 / 1000
correlation_puissance = X_train["puissance_kW"].corr(puissance_calculee)
print("corrélation de puissance_kW avec couple x vitesse :", round(correlation_puissance, 4))

Lecture du résultat : la médiane de l'âge passe de 10,1 ans pour la classe faible à 19,2 ans pour la classe élevée, celle des heures depuis le graissage de 1617 à 3236 h, la vibration de 1,6 à 2,7 mm/s : le risque monte avec ces relevés. La vitesse se comporte autrement : sa médiane est de 2276 tr/min pour la classe faible et de 1968 pour la classe élevée, mais l'écart entre quartiles se resserre de 1756 à 425 tr/min : les équipements à risque élevé tournent dans une bande étroite de vitesses, ce qu'aucun effet croissant ou décroissant de la vitesse ne représente. La puissance est le produit du couple et de la vitesse (corrélation 1,000 avec ce produit) : elle ne dit rien de plus.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression logistique sur tous les relevés. Le type d'équipement devient des colonnes de 0 et de 1, les cases vides de la contamination et de la température sont remplacées par la médiane de l'apprentissage, et chaque relevé numérique est ramené à une moyenne nulle et à un écart type de 1. La préparation s'apprend sur `X_train` seul et s'applique ensuite au test. Deux règles sans apprentissage le suivent : toujours « faible », et une règle de vibration dont les seuils sont les quantiles de l'apprentissage (62 % et 91 % des valeurs, les parts des classes).

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle classe des réducteurs qu'il n'a jamais vus. On lit le F-score macro, le rappel de chaque classe (la part des équipements d'une classe que le modèle retrouve) et la matrice de confusion. Les deux règles sans apprentissage donnent le point de comparaison.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, recall_score

classes = ["faible", "moyen", "eleve"]
y_pred = modele.predict(X_test)
score_depart = f1_score(y_test, y_pred, average="macro")
rappels = dict(zip(classes, recall_score(y_test, y_pred, labels=classes, average=None)))
score_faible = f1_score(y_test, np.full(len(y_test), "faible"), average="macro")
seuil_moyen, seuil_eleve = X_train["vibration_mm_s"].quantile([0.62, 0.91])
regle = np.where(X_test["vibration_mm_s"] > seuil_eleve, "eleve", np.where(X_test["vibration_mm_s"] > seuil_moyen, "moyen", "faible"))
score_regle = f1_score(y_test, regle, average="macro")
print(f"F-score macro : {score_depart:.3f}")
print("rappel par classe :", {k: round(float(r), 3) for k, r in rappels.items()})
print(f"F-score macro, toujours faible : {score_faible:.3f}")
print(f"F-score macro, règle de vibration (seuils {seuil_moyen:.2f} et {seuil_eleve:.2f} mm/s) : {score_regle:.3f}")
print("matrice de confusion (lignes : réel, colonnes : prédit ; faible, moyen, élevé)")
print(confusion_matrix(y_test, y_pred, labels=classes))

Lecture du résultat : le F-score macro est de 0,695, contre 0,255 pour la réponse constante « faible » et 0,448 pour la règle de vibration. Le modèle retrouve 87 % des équipements à risque faible, 57 % de ceux à risque moyen et 61 % de ceux à risque élevé : la classe rare est la moins bien retrouvée, et c'est la plus coûteuse à manquer. La vibration seule ne suffit pas : elle n'est qu'une faible trace de l'usure.

## Pour aller plus loin : une variable de résonance, puis des modèles non linéaires

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première ajoute une variable lue à l'étape 3 : l'écart entre la vitesse et 2 000 tr/min, centre de la bande étroite des équipements à risque élevé. Elle compare, par validation croisée stratifiée sur l'apprentissage seul, la régression logistique avec et sans cette variable, la même avec des poids de classes équilibrés, un arbre de profondeur 4, une forêt aléatoire et un gradient boosting. La seconde mesure le modèle retenu une seule fois sur le test.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : en validation croisée, la régression logistique de départ atteint 0,706 ; avec la variable de résonance, 0,811 ; avec des poids de classes équilibrés, 0,783 ; l'arbre 0,644, la forêt 0,807 et le gradient boosting 0,830 : le modèle retenu est « gradient boosting par histogrammes ». Mesuré une seule fois sur le test, il donne un F-score macro de 0,812, soit 0,117 de plus que le modèle de départ, avec un rappel de 69 % sur la classe élevée. Le gain vient surtout de la variable lue à l'étape 3 : elle fait passer la régression logistique de 0,706 à 0,811.

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Sur trois classes déséquilibrées, l'exactitude ne dit rien : la réponse constante « faible » donne 0,255 de F-score macro. Le stratifié garantit 36 cas de la classe élevée dans le test.
2. Lire l'apprentissage par classe avant de modéliser a révélé un effet que la régression logistique ne représente pas : une bande étroite de vitesses pour les équipements à risque élevé.
3. Une variable construite à partir de cette lecture, validée par validation croisée sur l'apprentissage, fait passer la régression logistique de 0,706 à 0,811.
4. Le modèle retenu, « gradient boosting par histogrammes », mesuré une seule fois sur le test, donne 0,812. Une classe de risque reste une estimation : seule une inspection qualifie l'équipement.